# Clase 206 — Data testing con Great Expectations + Pandera

Definimos un suite de expectations sobre California Housing y validamos con (a) Great Expectations 1.x, (b) Pandera, (c) Polars + checks ad-hoc. Inyectamos un bug a propósito para ver el sistema detectarlo.

In [ ]:
import pandas as pd, numpy as np
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.data.copy()
df['target'] = data.target
print(df.shape, '\n', df.describe().round(2).T[['min', 'max', 'mean']])

## 1. Great Expectations 1.x (in-process)

In [ ]:
# Requiere: pip install great-expectations>=1.0
try:
    import great_expectations as gx
    ctx = gx.get_context(mode='ephemeral')
    src = ctx.data_sources.add_pandas('demo')
    asset = src.add_dataframe_asset(name='housing')
    batch_def = asset.add_batch_definition_whole_dataframe('all')
    batch = batch_def.get_batch(batch_parameters={'dataframe': df})

    from great_expectations.expectations import (
        ExpectColumnValuesToNotBeNull, ExpectColumnValuesToBeBetween,
        ExpectTableRowCountToBeBetween, ExpectColumnPairValuesAToBeGreaterThanB,
    )
    suite = ctx.suites.add(gx.ExpectationSuite(name='housing_v1'))
    suite.add_expectation(ExpectTableRowCountToBeBetween(min_value=1000, max_value=100000))
    suite.add_expectation(ExpectColumnValuesToNotBeNull(column='MedInc'))
    suite.add_expectation(ExpectColumnValuesToBeBetween(column='MedInc', min_value=0, max_value=20))
    suite.add_expectation(ExpectColumnValuesToBeBetween(column='HouseAge', min_value=0, max_value=200))
    suite.add_expectation(ExpectColumnPairValuesAToBeGreaterThanB(column_A='AveRooms', column_B='AveBedrms'))

    result = batch.validate(suite)
    print(f'success: {result.success}, statistics: {result.statistics}')
    for r in result.results:
        flag = '✅' if r.success else '❌'
        print(f'{flag} {r.expectation_config.type}')
except ImportError:
    print('pip install great-expectations para esta celda')

## 2. Pandera — sintaxis más concisa

In [ ]:
import pandera as pa
from pandera import Column, Check, DataFrameSchema

schema = DataFrameSchema({
    'MedInc': Column(float, [Check.ge(0), Check.le(20), Check.not_null()]),
    'HouseAge': Column(float, [Check.ge(0), Check.le(200)]),
    'AveRooms': Column(float, Check.ge(0)),
    'AveBedrms': Column(float, Check.ge(0)),
    'target': Column(float, Check.ge(0)),
}, checks=[
    Check(lambda d: (d['AveRooms'] >= d['AveBedrms']).all(), name='rooms_ge_bedrooms'),
    Check(lambda d: len(d) > 1000, name='min_rows'),
])

validated = schema.validate(df, lazy=True)
print('OK — todas las checks pasaron')
print('shape:', validated.shape)

## 3. Inyectar un bug y ver el sistema detectarlo

In [ ]:
df_corrupt = df.copy()
df_corrupt.loc[df_corrupt.sample(50, random_state=1).index, 'MedInc'] = np.nan
df_corrupt.loc[df_corrupt.sample(10, random_state=2).index, 'HouseAge'] = -5
df_corrupt.loc[df_corrupt.sample(20, random_state=3).index, 'AveBedrms'] = 999   # más que AveRooms

try:
    schema.validate(df_corrupt, lazy=True)
    print('UNEXPECTED: validation passed')
except pa.errors.SchemaErrors as e:
    print(f'❌ {len(e.failure_cases)} check failures (esperado):')
    print(e.failure_cases.head(10).to_string(index=False))

## 4. Integración con pipeline — gate por exit code

In [ ]:
validate_script = '''\
# validate.py — corre como step del DVC/Airflow/Prefect/CI
import sys, json, pandas as pd, pandera as pa
from schemas import housing_schema   # mismo schema en módulo compartido

df = pd.read_parquet(sys.argv[1])
try:
    housing_schema.validate(df, lazy=True)
    print(json.dumps({"status": "ok", "rows": len(df)}))
    sys.exit(0)
except pa.errors.SchemaErrors as e:
    print(json.dumps({"status": "failed", "errors": e.failure_cases.head(20).to_dict(orient="records")}))
    sys.exit(1)   # exit != 0 → pipeline aborta
'''
print(validate_script)

## Ejercicio guiado

1. Agregá una expectation: `target` tiene que tener `mean` entre `last_week_mean ± 10%` (gate de drift relativo).
2. Configurá Great Expectations con un Data Docs HTML. Hospedalo en GitHub Pages desde un GH Actions workflow.
3. Convertí el `validate.py` en un step del DVC pipeline (Clase 194) que corre antes del training. Verificá que si la validación falla, `dvc repro` aborta.
4. Para datasets grandes, instalá PyDeequ + Spark y replicá el suite. Compará performance.
5. Suite separado en críticas (abortan pipeline) vs warnings (loggean pero no abortan).

## Conclusiones

- Data testing es ortogonal a unit testing: el código puede estar OK pero la data llegar rota.
- Pandera es más DX-friendly para Python puro; GE es más completo para auditoría.
- El suite es contrato: va a git, va por PR.
- Validation gate antes de training evita que un mes de retrainings de data sucia llegue a producción.